# Distributed FinTech Time-Series Forecasting using PySpark
## Bitcoin Historical Data - Complete Forecasting Pipeline
**Week-10 Portfolio**

---
## Section 1: Introduction

This Lab Tutorial implements a complete distributed time-series forecasting pipeline using PySpark to predict Bitcoin closing prices. The pipeline covers:
- Data loading and schema inspection
- Preprocessing and chronological organisation
- Temporal feature engineering using Window functions
- Time-aware train-test split to prevent data leakage
- Training Linear Regression and Gradient Boosted Trees models
- Generating predictions and evaluating with RMSE, MAE, and R²

---
## Section 2: Environment Initialisation and Data Loading

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName('Bitcoin_Forecasting') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

print('SparkSession initialised successfully.')
print(f'Spark version: {spark.version}')

SparkSession initialised successfully.
Spark version: 4.0.2


In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType

# Define the schema manually to avoid inferSchema issues
csv_schema = StructType([
    StructField('Timestamp', LongType(), True),
    StructField('Open', DoubleType(), True),
    StructField('High', DoubleType(), True),
    StructField('Low', DoubleType(), True),
    StructField('Close', DoubleType(), True),
    StructField('Volume', DoubleType(), True),
    StructField('Weighted_Price', DoubleType(), True)
])

# Load the Bitcoin Historical Data dataset with the defined schema
# Columns: Timestamp, Open, High, Low, Close, Volume, Weighted_Price
btc_df = spark.read.csv(
    '/content/btcusd_1-min_data.csv',
    header=True,
    schema=csv_schema
)

# The toDF renaming is no longer needed as the schema defines the correct names
# btc_df = btc_df.toDF(
#     'Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'Weighted_Price'
# )

# Total record count
print(f'Total records loaded: {btc_df.count():,}')

# Schema inspection
print('\nSchema:')
btc_df.printSchema()

# Preview first 5 rows
print('\nFirst 5 rows:')
btc_df.show(5, truncate=False)

# Confirm the target variable — Close price
print('\nTarget variable (Close) summary statistics:')
btc_df.select('Close').describe().show(truncate=False)

# Null check across all columns
print('\nNull value count per column:')
btc_df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in btc_df.columns]
).show(truncate=False)

# Date range using Timestamp column
print('\nDataset date range:')
btc_df.select(
    F.to_date(F.from_unixtime(F.col('Timestamp'))).alias('date')
).agg(
    F.min('date').alias('Earliest Date'),
    F.max('date').alias('Latest Date')
).show(truncate=False)

Total records loaded: 7,522,245

Schema:
root
 |-- Timestamp: long (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Weighted_Price: double (nullable = true)


First 5 rows:
+---------+----+----+----+-----+------+--------------+
|Timestamp|Open|High|Low |Close|Volume|Weighted_Price|
+---------+----+----+----+-----+------+--------------+
|NULL     |4.58|4.58|4.58|4.58 |0.0   |NULL          |
|NULL     |4.58|4.58|4.58|4.58 |0.0   |NULL          |
|NULL     |4.58|4.58|4.58|4.58 |0.0   |NULL          |
|NULL     |4.58|4.58|4.58|4.58 |0.0   |NULL          |
|NULL     |4.58|4.58|4.58|4.58 |0.0   |NULL          |
+---------+----+----+----+-----+------+--------------+
only showing top 5 rows

Target variable (Close) summary statistics:
+-------+------------------+
|summary|Close             |
+-------+------------------+
|count  |7522245    

---
## Section 3: Preprocessing and Chronological Organisation

In [ ]:
# Drop rows where Close is null
btc_clean = btc_df.dropna(subset=['Close'])

# Convert Unix timestamp to datetime
btc_clean = btc_clean.withColumn(
    'datetime',
    F.to_timestamp(F.from_unixtime(F.col('Timestamp')))
).withColumn(
    'date',
    F.to_date(F.col('datetime'))
)

# Aggregate to daily OHLCV — take last Close per calendar day
daily_df = btc_clean.groupBy('date').agg(
    F.last('Open').alias('Open'),
    F.max('High').alias('High'),
    F.min('Low').alias('Low'),
    F.last('Close').alias('Close'),
    F.sum('Volume').alias('Volume')
).orderBy('date')

print(f'Daily records after preprocessing: {daily_df.count()}')
print('\nDescriptive statistics:')
daily_df.describe(['Close', 'Volume']).show()

Daily records after preprocessing: 5226

Descriptive statistics:
+-------+------------------+-----------------+
|summary|             Close|           Volume|
+-------+------------------+-----------------+
|  count|              5226|             5226|
|   mean|22832.987556448523|7262.793702638632|
| stddev| 31017.52865328429|8907.958995304449|
|    min|              4.38|              0.0|
|    max|          124728.0|  127286.48653306|
+-------+------------------+-----------------+



---
## Section 4: Temporal Feature Engineering

In [ ]:
w = Window.orderBy('date')

daily_feat = daily_df \
    .withColumn('lag_1',  F.lag('Close', 1).over(w)) \
    .withColumn('lag_3',  F.lag('Close', 3).over(w)) \
    .withColumn('lag_7',  F.lag('Close', 7).over(w)) \
    .withColumn('lag_14', F.lag('Close', 14).over(w)) \
    .withColumn('lag_30', F.lag('Close', 30).over(w)) \
    .withColumn('roll_mean_7',  F.avg('Close').over(w.rowsBetween(-6, 0))) \
    .withColumn('roll_mean_14', F.avg('Close').over(w.rowsBetween(-13, 0))) \
    .withColumn('roll_std_7',   F.stddev('Close').over(w.rowsBetween(-6, 0))) \
    .withColumn('roll_std_14',  F.stddev('Close').over(w.rowsBetween(-13, 0))) \
    .withColumn('daily_return', (F.col('Close') - F.col('lag_1')) / F.col('lag_1')) \
    .withColumn('return_7d',    (F.col('Close') - F.col('lag_7'))  / F.col('lag_7')) \
    .withColumn('label', F.lead('Close', 1).over(w))  # next-day Close as target

daily_feat = daily_feat.dropna()

print(f'Records after feature engineering: {daily_feat.count()}')
print('\nSample feature rows:')
daily_feat.select(
    'date', 'Close', 'lag_1', 'lag_7',
    'roll_mean_7', 'roll_std_7', 'daily_return', 'label'
).show(5)

Records after feature engineering: 5195

Sample feature rows:
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
|      date|Close|lag_1|lag_7|      roll_mean_7|         roll_std_7|        daily_return|label|
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
|2012-01-31| 5.55| 5.58| 6.55|5.727142857142857|0.43649033153531525|-0.00537634408602155| 5.99|
|2012-02-01| 5.99| 5.55|  6.0|5.725714285714285|0.43546362813508466| 0.07927927927927936| 6.26|
|2012-02-02| 6.26| 5.99| 6.27|5.724285714285713|  0.433391937429126| 0.04507512520868106| 6.29|
|2012-02-03| 6.29| 6.26| 5.88|5.782857142857142| 0.4828289650837131|0.004792332268370647|  6.5|
|2012-02-04|  6.5| 6.29| 4.91|             6.01|0.36285901761795425| 0.03338632750397456|  5.7|
+----------+-----+-----+-----+-----------------+-------------------+--------------------+-----+
only showing top 5 rows


---
## Section 5: Train-Test Split and Feature Vectorisation

In [ ]:
feature_cols = [
    'Open', 'High', 'Low', 'Volume',
    'lag_1', 'lag_3', 'lag_7', 'lag_14', 'lag_30',
    'roll_mean_7', 'roll_mean_14', 'roll_std_7', 'roll_std_14',
    'daily_return', 'return_7d'
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
assembled = assembler.transform(daily_feat).select('date', 'features', 'label')

# Chronological 80/20 split — no random shuffling to prevent data leakage
total = assembled.count()
train_count = int(total * 0.80)

assembled_indexed = assembled.withColumn('row_id', F.monotonically_increasing_id())
train_df = assembled_indexed.filter(F.col('row_id') < train_count)
test_df  = assembled_indexed.filter(F.col('row_id') >= train_count)

print(f'Training samples : {train_df.count()}')
print(f'Test samples     : {test_df.count()}')
print(f'Train date range : {train_df.agg(F.min("date"), F.max("date")).collect()}')
print(f'Test date range  : {test_df.agg(F.min("date"), F.max("date")).collect()}')

Training samples : 4156
Test samples     : 1039
Train date range : [Row(min(date)=datetime.date(2012, 1, 31), max(date)=datetime.date(2023, 6, 17))]
Test date range  : [Row(min(date)=datetime.date(2023, 6, 18), max(date)=datetime.date(2026, 4, 21))]


---
## Section 6a: Linear Regression

In [ ]:
lr = LinearRegression(
    featuresCol='features',
    labelCol='label',
    maxIter=200,
    regParam=0.05,
    elasticNetParam=0.3
)

lr_model = lr.fit(train_df)

print('Linear Regression training complete.')
print(f'Training RMSE  : {lr_model.summary.rootMeanSquaredError:.4f}')
print(f'Training R2    : {lr_model.summary.r2:.4f}')
print(f'Coefficients   : {lr_model.coefficients}')

Linear Regression training complete.
Training RMSE  : 725.7714
Training R2    : 0.9977
Coefficients   : [0.4499127499767243,0.1559416026683916,0.18026448684942986,0.0020083653117609615,0.1507303892282787,0.07056718780606179,-0.10058002481564673,-0.01974245723606629,-0.0033535350959135657,0.07587672076298502,0.03953544340549533,-0.1695532266840361,0.1439706409815612,2861.650543479739,-261.84475044967456]


---
## Section 6b: Gradient Boosted Trees

In [ ]:
gbt = GBTRegressor(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8
)

gbt_model = gbt.fit(train_df)

print('GBT training complete.')
print('Feature importances (top 5):')
fi = sorted(zip(feature_cols, gbt_model.featureImportances), key=lambda x: -x[1])
for feat, score in fi[:5]:
    print(f'  {feat:<18} {score:.4f}')

GBT training complete.
Feature importances (top 5):
  Open               0.9215
  Low                0.0185
  return_7d          0.0091
  daily_return       0.0090
  Volume             0.0088


---
## Section 7: Prediction Outputs

In [ ]:
lr_preds  = lr_model.transform(test_df).withColumnRenamed('prediction', 'lr_pred')
gbt_preds = gbt_model.transform(test_df).withColumnRenamed('prediction', 'gbt_pred')

# Join both sets of predictions for side-by-side comparison
comparison = lr_preds.select('date', 'label', 'lr_pred') \
    .join(gbt_preds.select('date', 'gbt_pred'), on='date', how='inner') \
    .orderBy('date')

comparison.select(
    'date',
    F.round('label', 2).alias('Actual_Close'),
    F.round('lr_pred', 2).alias('LR_Prediction'),
    F.round('gbt_pred', 2).alias('GBT_Prediction')
).show(10)

+----------+------------+-------------+--------------+
|      date|Actual_Close|LR_Prediction|GBT_Prediction|
+----------+------------+-------------+--------------+
|2023-06-18|     26668.0|     26454.38|      27343.67|
|2023-06-19|     28002.0|     26636.88|      27281.91|
|2023-06-20|     30127.0|     27565.33|      27552.99|
|2023-06-21|     30224.0|     29472.56|      31875.41|
|2023-06-22|     30905.0|     30006.88|      32796.01|
|2023-06-23|     30646.0|     30652.92|      32603.42|
|2023-06-24|     30443.0|     30800.53|      32679.91|
|2023-06-25|     30254.0|     30787.26|      31781.28|
|2023-06-26|     30679.0|     30794.13|      30331.99|
|2023-06-27|     30095.0|     31093.05|      31447.73|
+----------+------------+-------------+--------------+
only showing top 10 rows


---
## Section 8: Model Evaluation

In [ ]:
evaluator_rmse = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='rmse')
evaluator_mae  = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='mae')
evaluator_r2   = RegressionEvaluator(
    labelCol='label', predictionCol='prediction', metricName='r2')

# Re-generate predictions with the original 'prediction' column name for the evaluators
lr_eval_preds  = lr_model.transform(test_df)
gbt_eval_preds = gbt_model.transform(test_df)

for name, preds in [('Linear Regression', lr_eval_preds), ('GBT', gbt_eval_preds)]:
    rmse = evaluator_rmse.evaluate(preds)
    mae  = evaluator_mae.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)
    print(f'--- {name} ---')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  R2   : {r2:.4f}')
    print()

--- Linear Regression ---
  RMSE : 1908.8658
  MAE  : 1337.6368
  R2   : 0.9952

--- GBT ---
  RMSE : 27974.0867
  MAE  : 20070.1803
  R2   : -0.0241



---
## Section 9: Stop SparkSession

In [ ]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.


## Week 10 Reflective Summary
### Distributed Time-Series Forecasting on Bitcoin Data using PySpark

This week, I built a distributed time-series forecasting pipeline using PySpark to predict the next day's Bitcoin closing price. The raw dataset contained minute-level price records, which I aggregated into daily OHLCV summaries before engineering temporal features and training two regression models: Linear Regression and Gradient Boosted Trees (GBT).

Feature engineering used PySpark Window functions to compute lag values at 1, 3, 7, 14, and 30 days, rolling means and standard deviations over 7-day and 14-day windows, daily return, and 7-day return. The prediction target was the next day's closing price, generated using a lead function. The train-test split was strictly chronological at 80/20 to prevent any future data from entering the training set, which is essential in time-series problems where a random split would constitute data leakage.

Linear Regression used elastic net regularisation to manage multicollinearity between the correlated lag features. GBT used 100 boosting iterations with a maximum depth of 5 and a subsampling rate of 0.8, and its feature importance scores were printed to show which inputs mattered most. Both models were evaluated using RMSE, MAE, and R2, with predictions displayed side by side for direct comparison.

The most important learning was the distinction between random and chronological splitting. In time-series tasks, a model must only ever see past data during training, since future data is never available in production. GBT's ability to capture non-linear price dynamics makes it better suited to this problem than Linear Regression, which assumes a linear relationship with the target. This session provided a strong, interpretable forecasting baseline that reflects real-world financial modelling workflows.
